In [ ]:
import os

# Search for chromadb folder in Drive
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for d in dirs:
        if "chroma" in d.lower() or "findebate" in d.lower():
            print(os.path.join(root, d))

/content/drive/MyDrive/findebate_chromadb


In [ ]:
# now we will start the part of mine, which is
# rag_part2.py

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install chromadb==0.5.23 sentence-transformers nltk -q
!pip install opentelemetry-api==1.41.1 opentelemetry-sdk==1.41.1 -q

In [ ]:
import chromadb
chroma_client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/findebate_chromadb"
)
collection = chroma_client.get_collection("findebate_rag")
print(f"Connected. Total chunks: {collection.count()}")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Connected. Total chunks: 6963


In [ ]:

import chromadb
from sentence_transformers import SentenceTransformer

# ── Connect to P1's existing ChromaDB ──
chroma_client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/findebate_chromadb"
)
collection = chroma_client.get_collection("findebate_rag")
model = SentenceTransformer("FinLang/finance-embeddings-investopedia")

# ── Verify connection ──
count = collection.count()
print(f"Connected. Total chunks: {count}")

# ── 4 Dimensions from Appendix B ──
DIMENSION_QUERIES = {
    "general_financial": [
        "financial performance revenue earnings beat miss surprise results",
        "guidance outlook forecast expectations future performance strategy",
        "growth trends margin expansion profitability cash flow competitive"
    ],
    "specialized_metrics": [
        "net interest margin NIM loan deposits credit quality asset quality",
        "non-performing assets NPAs charge-offs provision loan losses",
        "return on assets ROA return on equity ROE efficiency ratio capital adequacy"
    ],
    "market_sentiment_risk": [
        "management confidence sentiment optimistic cautious positive negative tone",
        "risks challenges concerns headwinds uncertainties market conditions",
        "credit risk operational risk market risk liquidity risk hedging"
    ],
    "multi_query_integration": [
        "short-term immediate near-term weekly monthly quarterly timeline",
        "comparative performance benchmarking cross-functional analysis",
        "comprehensive integrated multi-dimensional longitudinal tracking"
    ]
}

# ── Agent to Dimension Mapping ──
AGENT_DIMENSION_MAP = {
    "earnings_agent"  : ["general_financial", "specialized_metrics"],
    "market_agent"    : ["general_financial", "multi_query_integration"],
    "sentiment_agent" : ["market_sentiment_risk"],
    "valuation_agent" : ["specialized_metrics", "general_financial"],
    "risk_agent"      : ["market_sentiment_risk", "specialized_metrics"]
}

def retrieve(query, top_k=5, doc_type_filter=None):
    q_emb = model.encode([query], convert_to_numpy=True).tolist()
    where_clause = {"type": doc_type_filter} if doc_type_filter else None

    results = collection.query(
        query_embeddings = q_emb,
        n_results        = top_k,
        where            = where_clause,
        include          = ["documents", "metadatas", "distances"]
    )

    output = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        output.append({
            "chunk"       : doc,
            "source_file" : meta["source_file"],
            "chunk_id"    : meta["chunk_index"],
            "type"        : meta["type"],
            "score"       : round(1 - dist, 4)
        })
    return output

def retrieve_by_dimension(dimension, doc_type_filter=None, top_k=5):
    if dimension not in DIMENSION_QUERIES:
        raise ValueError(f"Unknown dimension: {dimension}")

    seen   = set()
    merged = []

    for query in DIMENSION_QUERIES[dimension]:
        results = retrieve(query, top_k=top_k, doc_type_filter=doc_type_filter)
        for r in results:
            uid = f"{r['source_file']}_chunk_{r['chunk_id']}"
            if uid not in seen:
                seen.add(uid)
                merged.append(r)

    merged.sort(key=lambda x: x["score"], reverse=True)
    return merged[:top_k]

def retrieve_all_dimensions(doc_type_filter=None, top_k=3):
    return {
        dim: retrieve_by_dimension(dim, doc_type_filter=doc_type_filter, top_k=top_k)
        for dim in DIMENSION_QUERIES
    }

def get_agent_context(source_file=None, top_k=3):
    context = {}

    for agent, dimensions in AGENT_DIMENSION_MAP.items():
        agent_chunks = []
        seen = set()

        for dim in dimensions:
            results = retrieve_by_dimension(dim, top_k=top_k*5)
            for r in results:
                if source_file and r["source_file"] != source_file:
                    continue
                uid = f"{r['source_file']}_chunk_{r['chunk_id']}"
                if uid not in seen:
                    seen.add(uid)
                    agent_chunks.append(r)

        agent_chunks.sort(key=lambda x: x["score"], reverse=True)
        context[agent] = agent_chunks[:top_k]

    return context

print("\n===== VALIDATION =====")

# Test 1 — basic retrieve
print("\n── Test 1: Basic Retrieve ──")
r = retrieve("revenue growth earnings", top_k=2)
for x in r:
    print(f"  [{x['score']}] {x['source_file']} | {x['chunk'][:100]}...")

# Test 2 — by dimension
print("\n── Test 2: Retrieve by Dimension ──")
for dim in DIMENSION_QUERIES:
    r = retrieve_by_dimension(dim, top_k=2)
    print(f"  {dim}: {len(r)} chunks retrieved")

# Test 3 — all dimensions
print("\n── Test 3: All Dimensions ──")
all_dims = retrieve_all_dimensions(top_k=2)
for dim, chunks in all_dims.items():
    print(f"  {dim}: {len(chunks)} chunks")

# Test 4 — agent context
print("\n── Test 4: Agent Context ──")
context = get_agent_context(top_k=2)
for agent, chunks in context.items():
    print(f"  {agent}: {len(chunks)} chunks")

print("\n===== VALIDATION COMPLETE =====")
print(f"Total chunks in ChromaDB: {collection.count()}")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Connected. Total chunks: 6963

===== VALIDATION =====

── Test 1: Basic Retrieve ──


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  [0.5717] CMI_q1_2015 | **Investor Relations** : Heavy-duty share....
  [0.5346] DNB_q2_2021 | ## Financial Earnings Call...

── Test 2: Retrieve by Dimension ──
  general_financial: 2 chunks retrieved
  specialized_metrics: 2 chunks retrieved
  market_sentiment_risk: 2 chunks retrieved
  multi_query_integration: 2 chunks retrieved

── Test 3: All Dimensions ──
  general_financial: 2 chunks
  specialized_metrics: 2 chunks
  market_sentiment_risk: 2 chunks
  multi_query_integration: 2 chunks

── Test 4: Agent Context ──
  earnings_agent: 2 chunks
  market_agent: 2 chunks
  sentiment_agent: 2 chunks
  valuation_agent: 2 chunks
  risk_agent: 2 chunks

===== VALIDATION COMPLETE =====
Total chunks in ChromaDB: 6963


In [1]:
import os
os.makedirs("slurm_scripts", exist_ok=True)
print("Folder created!")

Folder created!


In [2]:
# so now i tested with the HPC using ssh shell, we need a
# file that has instructions for the super computer, but since debugging is
# hard to do in the terminal, the GUI is **, so i am writing the code here

setup_env = """#!/bin/bash
# setup_env.sh
# Run this ONCE on HPC to install all dependencies

echo "Setting up FinDebate environment..."

# Load Python module
module load python/3.10

# Create virtual environment
python3 -m venv ~/findebate/venv
source ~/findebate/venv/bin/activate

# Install all required packages
pip install chromadb==0.5.23
pip install sentence-transformers
pip install nltk
pip install google-generativeai
pip install opentelemetry-api==1.41.1
pip install opentelemetry-sdk==1.41.1

echo "Environment setup complete!"
"""

with open("slurm_scripts/setup_env.sh", "w") as f:
    f.write(setup_env)

print("setup_env.sh created!")

setup_env.sh created!


In [3]:
pipeline = """#!/bin/bash
#SBATCH --job-name=findebate_pipeline
#SBATCH --partition=gpu_student
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=4
#SBATCH --gres=gpu:a100_1g.5gb:1
#SBATCH --mem=16G
#SBATCH --time=06:00:00
#SBATCH --output=/dgxa_home/se23ucse176/findebate/logs/pipeline_%j.log
#SBATCH --error=/dgxa_home/se23ucse176/findebate/logs/pipeline_%j.err

echo "Starting FinDebate Pipeline..."

# Activate environment
source ~/findebate/venv/bin/activate

# Move to project directory
cd ~/findebate

# Run the full pipeline
python3 pipeline.py

echo "Pipeline complete!"
"""

with open("slurm_scripts/findebate_pipeline.sh", "w") as f:
    f.write(pipeline)

print("findebate_pipeline.sh created!")

findebate_pipeline.sh created!


In [4]:
eval_grid = """#!/bin/bash
#SBATCH --job-name=findebate_eval
#SBATCH --partition=gpu_student
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=4
#SBATCH --gres=gpu:a100_1g.5gb:1
#SBATCH --mem=16G
#SBATCH --time=06:00:00
#SBATCH --output=/dgxa_home/se23ucse176/findebate/logs/eval_%j.log
#SBATCH --error=/dgxa_home/se23ucse176/findebate/logs/eval_%j.err

echo "Starting FinDebate Evaluation Grid..."

source ~/findebate/venv/bin/activate
cd ~/findebate

# Run evaluation across all 5 models
for MODEL in gpt-4o gemini-2.5-flash llama-4-maverick deepseek-r1 claude-sonnet-4
do
    echo "Evaluating with $MODEL..."
    python3 evaluate.py --model $MODEL
done

echo "Evaluation complete!"
"""

with open("slurm_scripts/eval_grid.sh", "w") as f:
    f.write(eval_grid)

print("eval_grid.sh created!")

eval_grid.sh created!


In [6]:
parallel_pipeline = """#!/bin/bash
#SBATCH --job-name=findebate_agents
#SBATCH --partition=gpu_student
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --gres=gpu:a100_1g.5gb:1
#SBATCH --mem=16G
#SBATCH --time=06:00:00
#SBATCH --array=0-4
#SBATCH --output=/dgxa_home/se23ucse176/findebate/logs/agent_%a_%j.log
#SBATCH --error=/dgxa_home/se23ucse176/findebate/logs/agent_%a_%j.err

# Array of all 5 agents
AGENTS=(earnings_agent market_agent sentiment_agent valuation_agent risk_agent)

# Pick the agent for this job based on array index
AGENT=${AGENTS[$SLURM_ARRAY_TASK_ID]}

echo "Starting agent: $AGENT"
echo "Array task ID: $SLURM_ARRAY_TASK_ID"

# Activate environment
source ~/findebate/venv/bin/activate

# Move to project directory
cd ~/findebate

# Run the specific agent
python3 agents/$AGENT.py

echo "$AGENT completed!"
"""

with open("slurm_scripts/findebate_pipeline.sh", "w") as f:
    f.write(parallel_pipeline)

print("Updated findebate_pipeline.sh with job array!")

Updated findebate_pipeline.sh with job array!


In [8]:
eval_grid = """#!/bin/bash
#SBATCH --job-name=findebate_eval
#SBATCH --partition=gpu_student
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --gres=gpu:a100_1g.5gb:1
#SBATCH --mem=16G
#SBATCH --time=06:00:00
#SBATCH --array=0-4
#SBATCH --output=/dgxa_home/se23ucse176/findebate/logs/eval_%a_%j.log
#SBATCH --error=/dgxa_home/se23ucse176/findebate/logs/eval_%a_%j.err

# All 5 models from the paper
MODELS=(gpt-4o gemini-2.5-flash llama-4-maverick deepseek-r1 claude-sonnet-4)

# Pick model for this job
MODEL=${MODELS[$SLURM_ARRAY_TASK_ID]}

echo "Evaluating with model: $MODEL"

source ~/findebate/venv/bin/activate
cd ~/findebate

python3 evaluate.py --model $MODEL

echo "$MODEL evaluation complete!"
"""

with open("slurm_scripts/eval_grid.sh", "w") as f:
    f.write(eval_grid)

print("Updated eval_grid.sh with job array!")

Updated eval_grid.sh with job array!


In [10]:
import os

print("===== SLURM SCRIPTS VERIFICATION =====\n")

for script in os.listdir("slurm_scripts"):
    filepath = f"slurm_scripts/{script}"
    with open(filepath, "r") as f:
        content = f.read()
    print(f"✅ {script}")
    print(f"   Size: {len(content)} characters")
    # Check if job array is present
    if "--array" in content:
        print(f"   Job Array: ✅ Present")
    else:
        print(f"   Job Array: ❌ Missing")
    print()

print("===== VERIFICATION COMPLETE =====")

===== SLURM SCRIPTS VERIFICATION =====

✅ setup_env.sh
   Size: 525 characters
   Job Array: ❌ Missing

✅ eval_grid.sh
   Size: 695 characters
   Job Array: ✅ Present

✅ findebate_pipeline.sh
   Size: 813 characters
   Job Array: ✅ Present

===== VERIFICATION COMPLETE =====


In [11]:
for script in os.listdir("slurm_scripts"):
    filepath = f"slurm_scripts/{script}"
    with open(filepath, "r") as f:
        content = f.read()
    print(f"\n===== {script} =====")
    print(content)


===== setup_env.sh =====
#!/bin/bash
# setup_env.sh
# Run this ONCE on HPC to install all dependencies

echo "Setting up FinDebate environment..."

# Load Python module
module load python/3.10

# Create virtual environment
python3 -m venv ~/findebate/venv
source ~/findebate/venv/bin/activate

# Install all required packages
pip install chromadb==0.5.23
pip install sentence-transformers
pip install nltk
pip install google-generativeai
pip install opentelemetry-api==1.41.1
pip install opentelemetry-sdk==1.41.1

echo "Environment setup complete!"


===== eval_grid.sh =====
#!/bin/bash
#SBATCH --job-name=findebate_eval
#SBATCH --partition=gpu_student
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --gres=gpu:a100_1g.5gb:1
#SBATCH --mem=16G
#SBATCH --time=06:00:00
#SBATCH --array=0-4
#SBATCH --output=/dgxa_home/se23ucse176/findebate/logs/eval_%a_%j.log
#SBATCH --error=/dgxa_home/se23ucse176/findebate/logs/eval_%a_%j.err

# All 5 models from the paper
MODELS=(gpt-4o gemini-2.5-flash l